# Practice - 12 - Advanced bridge: change data feed and SCD reasoning

Prerequisites / execution context: Run 00. Dedicated table. Uses the explicitly enabled legacy Delta CDF path for a controlled lab; current docs also describe automatic CDF with separate requirements.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R68: Change data feed](https://docs.databricks.com/aws/en/tables/features/change-data-feed); [R69: AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc); [R26: Delta MERGE](https://docs.databricks.com/aws/en/delta/merge); [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Observe Delta row changes

Create or replace SILVER.cdf_lab with id INT,name STRING and explicitly enabled legacy CDF. Capture start version before writes. Insert (1,old),(2,keep), update id=1 to new, delete id=2. Read table_changes from start, inspect commit/change metadata and store the set of observed change-type strings as types.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
T = f"{SILVER}.cdf_lab"

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert {"insert","update_preimage","update_postimage","delete"} <= types
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIEVuYWJsZSBDREYgYmVmb3JlIHdyaXRlcwpDREYgZXhwb3NlcyBjaGFuZ2VzIGZyb20gdGhlIHN1cHBvcnRlZCByZXRhaW5lZCBoaXN0b3J5LCBub3QgYSBwZXJtYW5lbnQgYXJjaGl2ZS4gUmVjb3JkIHRoZSBzdGFydCB2ZXJzaW9uIGZvciB5b3VyIGNvbnN1bWVyLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlQ9ZiZxdW90O3tTSUxWRVJ9LmNkZl9sYWImcXVvdDsKc3Bhcmsuc3FsKGYmcXVvdDtDUkVBVEUgT1IgUkVQTEFDRSBUQUJMRSB7VH0gKGlkIElOVCxuYW1lIFNUUklORykgVVNJTkcgREVMVEEgVEJMUFJPUEVSVElFUyAoZGVsdGEuZW5hYmxlQ2hhbmdlRGF0YUZlZWQ9dHJ1ZSkmcXVvdDspCnN0YXJ0PXNwYXJrLnNxbChmJnF1b3Q7REVTQ1JJQkUgSElTVE9SWSB7VH0gTElNSVQgMSZxdW90OykuZmlyc3QoKS52ZXJzaW9uCnNwYXJrLnNxbChmJnF1b3Q7SU5TRVJUIElOVE8ge1R9IFZBTFVFUyAoMSwmI3gyNztvbGQmI3gyNzspLCgyLCYjeDI3O2tlZXAmI3gyNzspJnF1b3Q7KQpzcGFyay5zcWwoZiZxdW90O1VQREFURSB7VH0gU0VUIG5hbWU9JiN4Mjc7bmV3JiN4Mjc7IFdIRVJFIGlkPTEmcXVvdDspCnNwYXJrLnNxbChmJnF1b3Q7REVMRVRFIEZST00ge1R9IFdIRVJFIGlkPTImcXVvdDspCmNoYW5nZXM9c3Bhcmsuc3FsKGYmcXVvdDtTRUxFQ1QgKiBGUk9NIHRhYmxlX2NoYW5nZXMoJiN4Mjc7e1R9JiN4Mjc7LHtzdGFydH0pJnF1b3Q7KQpjaGFuZ2VzLm9yZGVyQnkoJnF1b3Q7X2NvbW1pdF92ZXJzaW9uJnF1b3Q7LCZxdW90O2lkJnF1b3Q7LCZxdW90O19jaGFuZ2VfdHlwZSZxdW90Oykuc2hvdyh0cnVuY2F0ZT1GYWxzZSkKdHlwZXM9e3JbMF0gZm9yIHIgaW4gY2hhbmdlcy5zZWxlY3QoJnF1b3Q7X2NoYW5nZV90eXBlJnF1b3Q7KS5kaXN0aW5jdCgpLmNvbGxlY3QoKX0KYXNzZXJ0IHsmcXVvdDtpbnNlcnQmcXVvdDssJnF1b3Q7dXBkYXRlX3ByZWltYWdlJnF1b3Q7LCZxdW90O3VwZGF0ZV9wb3N0aW1hZ2UmcXVvdDssJnF1b3Q7ZGVsZXRlJnF1b3Q7fSAmbHQ7PSB0eXBlcwo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Written / design task 1

Model an IT -> FR customer change in SCD Type 1 and Type 2. Define keys, ordering and non-overlapping validity boundaries.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMi4gU0NEIFR5cGUgMSB2ZXJzdXMgVHlwZSAyClR5cGUgMSBrZWVwcyBjdXJyZW50IGF0dHJpYnV0ZXMuIFR5cGUgMiByZXRhaW5zIGVmZmVjdGl2ZSBwZXJpb2RzIGZvciBhdHRyaWJ1dGUgdmVyc2lvbnMuIEZvciBhIGN1c3RvbWVyIG1vdmluZyBJVCAtJmd0OyBGUiwgc2tldGNoIG9uZSBjdXJyZW50IFR5cGUgMSByb3cgYW5kIHR3byBUeXBlIDIgcm93cyB3aXRoIG5vbi1vdmVybGFwcGluZyB2YWxpZGl0eSBpbnRlcnZhbHMuIFVzZSBhIHNvdXJjZSBzZXF1ZW5jZSBvciBldmVudCB0aW1lc3RhbXAgcGx1cyBhIGRldGVybWluaXN0aWMgdGllLWJyZWFrZXIsIG5vdCBmaWxlLWFycml2YWwgb3JkZXIuIFJldmlldyBwaXBlbGluZXMvY3VzdG9tZXJfY2RjLnB5IGZvciBBVVRPIENEQyB3aXRoIFNDRCBUeXBlIDIuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9sZHAvY2RjIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjY5OiBBVVRPIENEQzwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')


## Coding task 2: Prepare an out-of-order CDC feed

Optionally create unique CDC_FEED in Bronze with id INT,country STRING,seq BIGINT,op STRING. Insert customer 1/IT at sequence 1. Print the configuration and SQL needed to later append FR at seq=3, then DE at seq=2, between pipeline updates.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
PREPARE_CDC_FEED = False

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIyBPcHRpb25hbCBmZWVkIHByZXBhcmF0aW9uIGZvciB0aGUgc2VwYXJhdGUgQVVUTyBDREMgcGlwZWxpbmUKVGhpcyBjcmVhdGVzIGEgdW5pcXVlIGFwcGVuZC1vbmx5IHN5bnRoZXRpYyBmZWVkLiBDb3B5IHRoZSBwcmludGVkIHRhYmxlIG5hbWUgaW50byBraXQuY2RjX3NvdXJjZSB3aGVuIGNvbmZpZ3VyaW5nIHRoZSBORVcgcGlwZWxpbmUuIEFmdGVyIHRoZSBmaXJzdCBwaXBlbGluZSB1cGRhdGUgc3VjY2VlZHMsIGFwcGVuZCBzZXF1ZW5jZSAzLCBydW4gYW5vdGhlciB1cGRhdGUsIHRoZW4gYXBwZW5kIHNlcXVlbmNlIDIgYW5kIHVwZGF0ZSBhZ2Fpbi4gVGhlIHNlcGFyYXRlIGNlbGxzIGFyZSBpbnRlbnRpb25hbDogdGhleSBsZXQgeW91IG9ic2VydmUgYXJyaXZhbCBvcmRlci4gPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9sZHAvY2RjIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+T2ZmaWNpYWwgQVVUTyBDREM8L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlBSRVBBUkVfQ0RDX0ZFRUQ9RmFsc2UKaWYgUFJFUEFSRV9DRENfRkVFRDoKICAgIGltcG9ydCB1dWlkCiAgICBDRENfRkVFRD1mJnF1b3Q7e0JST05aRX0uY3VzdG9tZXJfZmVlZF97dXVpZC51dWlkNCgpLmhleFs6MTBdfSZxdW90OwogICAgc3Bhcmsuc3FsKGYmcXVvdDtDUkVBVEUgVEFCTEUge0NEQ19GRUVEfSAoaWQgSU5ULGNvdW50cnkgU1RSSU5HLHNlcSBCSUdJTlQsb3AgU1RSSU5HKSBVU0lORyBERUxUQSZxdW90OykKICAgIHNwYXJrLnNxbChmJnF1b3Q7SU5TRVJUIElOVE8ge0NEQ19GRUVEfSBWQUxVRVMgKDEsJiN4Mjc7SVQmI3gyNzssMSwmI3gyNztJTlNFUlQmI3gyNzspJnF1b3Q7KQogICAgcHJpbnQoJnF1b3Q7a2l0LmNkY19zb3VyY2UgPSZxdW90OyxDRENfRkVFRCkKICAgIHByaW50KCZxdW90O0FmdGVyIHBpcGVsaW5lIHVwZGF0ZSwgcnVuOiZxdW90OywgZiZxdW90O0lOU0VSVCBJTlRPIHtDRENfRkVFRH0gVkFMVUVTICgxLCYjeDI3O0ZSJiN4Mjc7LDMsJiN4Mjc7VVBEQVRFJiN4Mjc7KSZxdW90OykKICAgIHByaW50KCZxdW90O0FmdGVyIHRoZSBuZXh0IHVwZGF0ZSwgcnVuOiZxdW90OywgZiZxdW90O0lOU0VSVCBJTlRPIHtDRENfRkVFRH0gVkFMVUVTICgxLCYjeDI3O0RFJiN4Mjc7LDIsJiN4Mjc7VVBEQVRFJiN4Mjc7KSZxdW90OykKICAgIHByaW50KCZxdW90O0luc3BlY3QgdGhlIHRhcmdldCBiZWZvcmUgdGVzdGluZyBhIGRlbGV0ZTsgc291cmNlIGFycml2YWwgb3JkZXIgZGlmZmVycyBmcm9tIHNlcSBvcmRlci4mcXVvdDspCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Written / design task 2

Build a separate AUTO CDC Type 2 pipeline over the synthetic feed. Specify source, target, key, sequence, delete handling and excluded control columns. Predict history after sequences 1,3,2.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDIgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgMy4gQVVUTyBDREMgZXh0ZW5zaW9uCkFmdGVyIGNyZWF0aW5nIGFuIGFwcGVuZC1vbmx5IENEQyBmZWVkIHRhYmxlIHdpdGggaWQsIGNvdW50cnksIHNlcSBhbmQgb3AsIGFkZCB0aGUgYWR2YW5jZWQgc291cmNlIHRvIGEgTkVXIHBpcGVsaW5lIGFuZCBzZXQga2l0LmNkY19zb3VyY2UgdG8gaXRzIGZ1bGx5IHF1YWxpZmllZCBuYW1lLiBSdW4gZXZlbnRzIElOU0VSVChpZD0xLGNvdW50cnk9SVQsc2VxPTEpLCBVUERBVEUoaWQ9MSxjb3VudHJ5PUZSLHNlcT0zKSwgdGhlbiBhbiBvbGRlciBVUERBVEUoc2VxPTIpIHRvIHN0dWR5IG9yZGVyaW5nLiBRdWVyeSBfX1NUQVJUX0FUIGFuZCBfX0VORF9BVCBhbmQgdmVyaWZ5IHRoYXQgY3VycmVudCBhdHRyaWJ1dGVzIGZvbGxvdyBzZXF1ZW5jZS4gVXNlIGEgZGVsZXRlIGV2ZW50IGFuZCBjb25maXJtIGN1cnJlbnQtcm93IHJlbW92YWwgd2hpbGUgaGlzdG9yeSBmb2xsb3dzIHRoZSBjb25maWd1cmVkIFNDRCBiZWhhdmlvci4gT2ZmaWNpYWwgRGF0YWJyaWNrcyBkb2N1bWVudGF0aW9uOiA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2xkcC9jZGMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjk6IEFVVE8gQ0RDPC9hPi4KPC9kaXY+PC9kZXRhaWxzPg==')


## Written / design task 3

Design a replay after a week-long outage, idempotent foreachBatch updates, a measured watermark policy, and a test for older late CDC events.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDMgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgNC4gUHJvZmVzc2lvbmFsLWxldmVsIGRpc2N1c3Npb24gcHJvbXB0cwpIb3cgZG8geW91IHJldGFpbiBlbm91Z2ggc291cmNlIGhpc3RvcnkgdG8gcmVwbGF5IGFmdGVyIGEgd2Vlay1sb25nIG91dGFnZT8gSG93IGRvIHlvdSBtYWtlIGEgZm9yZWFjaEJhdGNoIE1FUkdFIGlkZW1wb3RlbnQgb24gcmV0cmllcz8gSG93IGRvIHlvdSBjaG9vc2Ugd2F0ZXJtYXJrIGRlbGF5IHVzaW5nIG9ic2VydmVkIGxhdGVuZXNzPyBXaGF0IHRlc3QgY2F0Y2hlcyBhIGxhdGUgb2xkZXIgQ0RDIGV2ZW50IG92ZXJ3cml0aW5nIGEgbmV3ZXIgdmFsdWU/IEhvdyBkbyB5b3UgY29tcGFyZSBzb3VyY2UgYW5kIHRhcmdldCBhdCB0aGUgc2FtZSBjdXRvZmY/IFdyaXRlIGEgZGVzaWduIGJlZm9yZSBjb2RpbmcuIFRoZXNlIHRvcGljcyBhcmUgYSBicmlkZ2UsIG5vdCBhIGNsYWltIGFib3V0IGEgZnV0dXJlIGNlcnRpZmljYXRpb24gYmx1ZXByaW50LiBPZmZpY2lhbCBEYXRhYnJpY2tzIGRvY3VtZW50YXRpb246IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vc3RydWN0dXJlZC1zdHJlYW1pbmcvY2hlY2twb2ludHMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMzA6IFN0cmVhbWluZyBjaGVja3BvaW50czwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vc3RydWN0dXJlZC1zdHJlYW1pbmcvd2F0ZXJtYXJrcyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIzMTogV2F0ZXJtYXJrczwvYT47IDxhIGhyZWY9Imh0dHBzOi8vZG9jcy5kYXRhYnJpY2tzLmNvbS9hd3MvZW4vdGFibGVzL2ZlYXR1cmVzL2NoYW5nZS1kYXRhLWZlZWQiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjg6IENoYW5nZSBkYXRhIGZlZWQ8L2E+OyA8YSBocmVmPSJodHRwczovL2RvY3MuZGF0YWJyaWNrcy5jb20vYXdzL2VuL2xkcC9jZGMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SNjk6IEFVVE8gQ0RDPC9hPi4KPC9kaXY+PC9kZXRhaWxzPg==')


## Optional supporting reference files
Try writing your own configuration/source first. These panels contain the full source needed for the external pipeline/Job tasks. Reveal, then save the code to the stated workspace file paths. They are not executed by opening a panel. Replace catalog/prefix settings with your practice configuration.


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgY3VzdG9tZXJfY2RjLnB5PC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPiMgQURWQU5DRUQ6IGFkZCB0byBhIE5FVyBwaXBlbGluZSB3aXRoIGtpdC5jZGNfc291cmNlIHNldCB0byBhbiBleGlzdGluZyBmZWVkLgojIEZlZWQgc2NoZW1hOiBpZCBJTlQsIGNvdW50cnkgU1RSSU5HLCBzZXEgTE9ORywgb3AgU1RSSU5HOyBvcCBERUxFVEUgaXMgYSB0b21ic3RvbmUuCiMgT2ZmaWNpYWwgZG9jczogaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9sZHAvY2RjCmZyb20gcHlzcGFyayBpbXBvcnQgcGlwZWxpbmVzIGFzIGRwCmZyb20gcHlzcGFyay5zcWwuZnVuY3Rpb25zIGltcG9ydCBjb2wsIGV4cHIKCkBkcC50YWJsZShuYW1lPSZxdW90O2N1c3RvbWVyX2NoYW5nZXMmcXVvdDspCmRlZiBjdXN0b21lcl9jaGFuZ2VzKCk6CiAgICByZXR1cm4gc3BhcmsucmVhZFN0cmVhbS50YWJsZShzcGFyay5jb25mLmdldCgmcXVvdDtraXQuY2RjX3NvdXJjZSZxdW90OykpCgpkcC5jcmVhdGVfc3RyZWFtaW5nX3RhYmxlKG5hbWU9JnF1b3Q7Y3VzdG9tZXJzX3NjZDImcXVvdDspCmRwLmNyZWF0ZV9hdXRvX2NkY19mbG93KAogICAgdGFyZ2V0PSZxdW90O2N1c3RvbWVyc19zY2QyJnF1b3Q7LCBzb3VyY2U9JnF1b3Q7Y3VzdG9tZXJfY2hhbmdlcyZxdW90Oywga2V5cz1bJnF1b3Q7aWQmcXVvdDtdLAogICAgc2VxdWVuY2VfYnk9Y29sKCZxdW90O3NlcSZxdW90OyksIGFwcGx5X2FzX2RlbGV0ZXM9ZXhwcigmcXVvdDtvcCA9ICYjeDI3O0RFTEVURSYjeDI3OyZxdW90OyksCiAgICBleGNlcHRfY29sdW1uX2xpc3Q9WyZxdW90O29wJnF1b3Q7LCAmcXVvdDtzZXEmcXVvdDtdLCBzdG9yZWRfYXNfc2NkX3R5cGU9MikKPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')
